In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="13vCTuQnP3PJxJJP5wfM3t8jc0rG_MQsk", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
import time

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. This notebook will use CPU.")
    print("   Go to Runtime → Change runtime type → GPU (recommended)")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

import matplotlib.pyplot as plt

# Clean plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'figure.dpi': 100,
})

%matplotlib inline

# 🔐 Gated DeltaNet: The Full Layer

*Part 3 of the Vizuara series on Building Qwen3.5 from Scratch*
*Estimated time: ~90 minutes*

In Notebook 2, we built the **delta rule** — an error-correcting memory update that surgically fixes what the state matrix gets wrong. But we also discovered its fatal limitation: it cannot **fully forget**. When the model needs to discard an entire context and start fresh, the delta rule alone is powerless.

In this notebook, we will:
1. **Add exponential gating** ($\alpha_t$, $\beta_t$) that gives the model a learnable "memory wipe button"
2. **Walk through the numerical example** from the article ($\alpha = 0.3$, $\beta = 0.9$) to see near-complete memory reset
3. **Build the complete GatedDeltaNet layer** with Conv1D, SiLU, L2 norm, and multi-head structure
4. **Benchmark inference cost** — O(1) per token vs standard attention's O(N) — and visualize the difference
5. **Connect everything to Qwen3.5** — these layers power 75% of the model

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/build-llm/practice/7/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

---


In [ ]:
#@title 🎧 Listen: Recap Delta Limitation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_recap_delta_limitation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. 🔄 Recap: The Delta Rule and Its Limitation

### Where We Left Off

In Notebook 2, we implemented the delta rule update:

$$S_t = S_{t-1} + k_t \left(v_t - S_{t-1}^T k_t\right)^T$$

This turned a dumb additive accumulator into an **error-correcting associative memory**. Instead of blindly writing the raw value $v_t$, the delta rule first checks what the memory currently predicts ($S_{t-1}^T k_t$), computes the error ($v_t - S_{t-1}^T k_t$), and writes only the correction.

The result was dramatic: DeltaNet achieved orders-of-magnitude lower retrieval error compared to naive linear attention.

### The Limitation We Discovered

But the delta rule has a blind spot: **it cannot perform wholesale forgetting**. It can correct individual key-value associations, but it cannot wipe the state matrix clean. When the model encounters a topic change — say, switching from discussing cooking to discussing quantum physics — the delta rule has no mechanism to say "forget everything about cooking."

We need a **decay gate**: a learnable scalar $\alpha_t \in (0, 1)$ that multiplies the entire state matrix at each step, allowing the model to decide how much of the past to retain.

In [ ]:
# Bring forward the delta rule from Notebook 2
def delta_rule_update(S: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Apply one step of the delta rule to the state matrix.

    S_new = S + k @ (v - S^T @ k)^T
    """
    prediction = S.T @ k
    error = v - prediction
    S_new = S + torch.outer(k, error)
    return S_new


# Quick check: it still works
S_0 = torch.tensor([[0.8, 0.2],
                     [0.1, 0.9]])
k_1 = torch.tensor([1.0, 0.0])
v_1 = torch.tensor([0.5, 0.3])

S_1 = delta_rule_update(S_0, k_1, v_1)
retrieved = S_1.T @ k_1
print(f"Delta rule sanity check:")
print(f"  S_1^T @ k_1 = {retrieved.tolist()}")
print(f"  Expected v_1 = {v_1.tolist()}")
print(f"  ✅ Match!" if torch.allclose(retrieved, v_1) else "  ❌ Mismatch!")

---


In [ ]:
#@title 🎧 Listen: Exponential Gating Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_exponential_gating_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. ⚡ Exponential Gating: The Memory Wipe Button

### The Gated DeltaNet Update Rule

The Gated DeltaNet adds two learnable scalars to the delta rule:

$$S_t = \alpha_t \cdot S_{t-1} + \beta_t \cdot k_t \left(v_t - S_{t-1}^T k_t\right)^T$$

| Symbol | Name | Range | Role |
|--------|------|-------|------|
| $\alpha_t$ | **Decay gate** | $(0, 1)$ | How much old memory to **retain** |
| $\beta_t$ | **Write strength** | $(0, 1)$ | How strongly to **write** the correction |

Both are computed from the input at each time step:

$$\alpha_t = \sigma(W_\alpha x_t + b_\alpha), \qquad \beta_t = \sigma(W_\beta x_t + b_\beta)$$

where $\sigma$ is the sigmoid function. This means the model **learns when to forget** and **how strongly to write**, conditioned on what it is currently reading.

### The Complementary Roles

Think of it this way:

- **Gating** ($\alpha_t$) is **wholesale erasure** — it scales the *entire* state matrix by a factor between 0 and 1. When $\alpha_t \approx 0$, the model says "forget everything." When $\alpha_t \approx 1$, it says "keep everything."

- **Delta rule** ($\beta_t \cdot k_t(v_t - S_{t-1}^T k_t)^T$) is **surgical correction** — it targets a specific key direction and writes only the error. This is fine-grained, not wholesale.

Together, they give the model two complementary tools:
1. A sledgehammer ($\alpha_t$) for clearing the whole whiteboard
2. A scalpel ($\beta_t$ + delta rule) for making precise corrections

### Special Cases

| $\alpha_t$ | $\beta_t$ | Behavior |
|------------|-----------|----------|
| 1 | 1 | Pure delta rule (Notebook 2) |
| 1 | 0 | No update — state passes through unchanged |
| 0 | 1 | Total memory reset — only the new write survives |
| 0.3 | 0.9 | Near-complete reset + strong new write (article example) |

In [ ]:
#@title 🎧 Listen: Worked Example Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_worked_example_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Worked Example: $\alpha = 0.3$, $\beta = 0.9$

Let us walk through the exact numerical example from the article. We start with a state matrix that has accumulated information:

$$S_0 = \begin{bmatrix} 0.8 & 0.2 \\ 0.1 & 0.9 \end{bmatrix}$$

A new token arrives with $k = [1, 0]^T$, $v = [0.5, 0.3]^T$, and the model produces gates $\alpha = 0.3$, $\beta = 0.9$.

**Step 1 — Decay the old state:**

$$\alpha \cdot S_0 = 0.3 \times \begin{bmatrix} 0.8 & 0.2 \\ 0.1 & 0.9 \end{bmatrix} = \begin{bmatrix} 0.24 & 0.06 \\ 0.03 & 0.27 \end{bmatrix}$$

The state has been **reduced to 30% of its previous magnitude**. This is the wholesale forgetting.

**Step 2 — Compute the error (using the decayed state for prediction):**

$$\text{prediction} = (\alpha \cdot S_0)^T k = \begin{bmatrix} 0.24 & 0.03 \\ 0.06 & 0.27 \end{bmatrix} \begin{bmatrix} 1 \\ 0 \end{bmatrix} = \begin{bmatrix} 0.24 \\ 0.06 \end{bmatrix}$$

$$\text{error} = v - \text{prediction} = \begin{bmatrix} 0.5 \\ 0.3 \end{bmatrix} - \begin{bmatrix} 0.24 \\ 0.06 \end{bmatrix} = \begin{bmatrix} 0.26 \\ 0.24 \end{bmatrix}$$

**Step 3 — Write the scaled correction:**

$$\beta \cdot k \cdot \text{error}^T = 0.9 \times \begin{bmatrix} 1 \\ 0 \end{bmatrix} \begin{bmatrix} 0.26 & 0.24 \end{bmatrix} = \begin{bmatrix} 0.234 & 0.216 \\ 0 & 0 \end{bmatrix}$$

**Step 4 — Combine:**

$$S_1 = \begin{bmatrix} 0.24 & 0.06 \\ 0.03 & 0.27 \end{bmatrix} + \begin{bmatrix} 0.234 & 0.216 \\ 0 & 0 \end{bmatrix} = \begin{bmatrix} 0.474 & 0.276 \\ 0.03 & 0.27 \end{bmatrix}$$

**Verify retrieval:**

$$S_1^T k = \begin{bmatrix} 0.474 & 0.03 \\ 0.276 & 0.27 \end{bmatrix} \begin{bmatrix} 1 \\ 0 \end{bmatrix} = \begin{bmatrix} 0.474 \\ 0.276 \end{bmatrix}$$

This is not exactly $v = [0.5, 0.3]^T$ — the aggressive decay ($\alpha = 0.3$) threw away so much old state that the error correction could not fully compensate in one step. But the key point is: **the old information has been almost entirely erased**. The Frobenius norm went from 1.25 down to 0.59 — a 53% reduction.

In [ ]:
#@title 🎧 Code Walkthrough: Worked Example Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_worked_example_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verify the worked example from the article
print("=" * 60)
print("  WORKED EXAMPLE: Gated DeltaNet (α=0.3, β=0.9)")
print("=" * 60)

S_0 = torch.tensor([[0.8, 0.2],
                     [0.1, 0.9]])
k = torch.tensor([1.0, 0.0])
v = torch.tensor([0.5, 0.3])
alpha = 0.3
beta = 0.9

print(f"\nInputs:")
print(f"  S_0 =\n{S_0}")
print(f"  k = {k.tolist()}, v = {v.tolist()}")
print(f"  α = {alpha}, β = {beta}")

# Step 1: Decay
decayed = alpha * S_0
print(f"\nStep 1 — Decay: α·S_0 =")
print(f"  {decayed}")

# Step 2: Error (using decayed state)
prediction = decayed.T @ k
error = v - prediction
print(f"\nStep 2 — Prediction: (α·S_0)^T @ k = {prediction.tolist()}")
print(f"         Error:      v - pred       = {error.tolist()}")

# Step 3: Scaled correction
correction = beta * torch.outer(k, error)
print(f"\nStep 3 — Correction: β · k ⊗ error =")
print(f"  {correction}")

# Step 4: Combine
S_1 = decayed + correction
print(f"\nStep 4 — S_1 = decayed + correction =")
print(f"  {S_1}")

# Verify retrieval
retrieved = S_1.T @ k
print(f"\nRetrieval: S_1^T @ k = {retrieved.tolist()}")
print(f"Target v:              {v.tolist()}")

# Check norms
norm_before = torch.norm(S_0).item()
norm_after = torch.norm(S_1).item()
print(f"\nFrobenius norm: {norm_before:.4f} → {norm_after:.4f} "
      f"({(1 - norm_after/norm_before)*100:.0f}% reduction)")
print(f"\n✨ The aggressive decay (α=0.3) wiped 70% of the old state,")
print(f"   then the write strength (β=0.9) wrote a strong new correction.")
print(f"   Old information has been nearly erased.")

In [ ]:
#@title 🎧 What to Look For: Visualize Decay Write
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_visualize_decay_write.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Visualizing the Decay-Write Tradeoff

Let us see how different combinations of $\alpha$ and $\beta$ affect the state. We will build up a state matrix from 10 random writes (using the delta rule), then apply a single gated update and measure the resulting Frobenius norm.

In [ ]:
# Visualize how alpha and beta affect the state
torch.manual_seed(42)
d = 8

# Build up some state using delta rule
S_base = torch.zeros(d, d)
for _ in range(10):
    k_rand = torch.randn(d)
    k_rand = k_rand / k_rand.norm()
    v_rand = torch.randn(d)
    S_base = delta_rule_update(S_base, k_rand, v_rand)

base_norm = torch.norm(S_base).item()
print(f"Base state Frobenius norm: {base_norm:.4f}")

# Try different alpha/beta combinations
k_new = torch.randn(d); k_new = k_new / k_new.norm()
v_new = torch.randn(d)

alphas = np.linspace(0, 1, 50)
betas = np.linspace(0, 1, 50)
A, B = np.meshgrid(alphas, betas)
norms = np.zeros_like(A)

for i in range(len(betas)):
    for j in range(len(alphas)):
        a, b = alphas[j], betas[i]
        decayed = a * S_base
        pred = decayed.T @ k_new
        err = v_new - pred
        S_new = decayed + b * torch.outer(k_new, err)
        norms[i, j] = torch.norm(S_new).item()

# Plot as heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Norm heatmap
im = axes[0].imshow(norms, origin='lower', aspect='auto',
                     extent=[0, 1, 0, 1], cmap='viridis')
axes[0].set_xlabel('α (decay gate)', fontsize=12)
axes[0].set_ylabel('β (write strength)', fontsize=12)
axes[0].set_title('State Norm After Gated Update', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[0], label='Frobenius Norm')

# Mark special cases
axes[0].plot(1.0, 1.0, 'w*', markersize=15, label='α=1, β=1 (delta rule)')
axes[0].plot(0.0, 1.0, 'r*', markersize=15, label='α=0, β=1 (total reset)')
axes[0].plot(0.3, 0.9, 'y*', markersize=15, label='α=0.3, β=0.9 (article)')
axes[0].legend(loc='lower right', fontsize=8, facecolor='white', framealpha=0.9)

# Right: Norm vs alpha for fixed beta values
for b_val in [0.1, 0.5, 0.9, 1.0]:
    norms_1d = []
    for a_val in alphas:
        decayed = a_val * S_base
        pred = decayed.T @ k_new
        err = v_new - pred
        S_n = decayed + b_val * torch.outer(k_new, err)
        norms_1d.append(torch.norm(S_n).item())
    axes[1].plot(alphas, norms_1d, linewidth=2, label=f'β={b_val}')

axes[1].axhline(y=base_norm, color='gray', linestyle='--', alpha=0.7, label='Original norm')
axes[1].set_xlabel('α (decay gate)', fontsize=12)
axes[1].set_ylabel('State Frobenius Norm', fontsize=12)
axes[1].set_title('How α Controls Memory Retention', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Left:  The heatmap shows how (α, β) jointly control the state magnitude.")
print("       Low α = aggressive forgetting. High β = strong writing.")
print("Right: For any β, lowering α reduces the state norm (more forgetting).")
print("       The model learns to set these gates based on the input.")

In [ ]:
#@title 🎧 What to Look For: State Evolution Gated Ungated
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_state_evolution_gated_ungated.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### State Evolution: Gated vs Ungated

Let us watch how the state matrix evolves over a sequence. We will simulate a scenario where the model processes two "contexts" (e.g., two different topics), with a topic switch in the middle. The gated version should be able to clear the old context at the switch point.

In [ ]:
# Simulate state evolution with and without gating
torch.manual_seed(42)
d = 8
seq_len = 40

# Generate two "contexts" — different random distributions
context_1_keys = torch.randn(seq_len // 2, d)
context_1_keys = context_1_keys / context_1_keys.norm(dim=1, keepdim=True)
context_1_values = torch.randn(seq_len // 2, d) * 2  # Stronger signal

context_2_keys = torch.randn(seq_len // 2, d)
context_2_keys = context_2_keys / context_2_keys.norm(dim=1, keepdim=True)
context_2_values = torch.randn(seq_len // 2, d) * 2

keys = torch.cat([context_1_keys, context_2_keys], dim=0)
values = torch.cat([context_1_values, context_2_values], dim=0)

# Generate gate values: high alpha during each context, low alpha at the switch
alphas_seq = torch.ones(seq_len)
alphas_seq[seq_len // 2] = 0.05     # Near-total reset at the context switch
alphas_seq[seq_len // 2 + 1] = 0.1  # Still low right after switch
betas_seq = torch.ones(seq_len) * 0.9  # Consistently strong writes

# Process: ungated (delta rule only)
S_ungated = torch.zeros(d, d)
norms_ungated = []
for t in range(seq_len):
    S_ungated = delta_rule_update(S_ungated, keys[t], values[t])
    norms_ungated.append(torch.norm(S_ungated).item())

# Process: gated
S_gated = torch.zeros(d, d)
norms_gated = []
for t in range(seq_len):
    a, b = alphas_seq[t].item(), betas_seq[t].item()
    decayed = a * S_gated
    pred = decayed.T @ keys[t]
    err = values[t] - pred
    S_gated = decayed + b * torch.outer(keys[t], err)
    norms_gated.append(torch.norm(S_gated).item())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: State norm over time
axes[0].plot(range(seq_len), norms_ungated, linewidth=2.5, color='#e53935',
             label='Ungated (Delta Rule)')
axes[0].plot(range(seq_len), norms_gated, linewidth=2.5, color='#1565c0',
             label='Gated DeltaNet')
axes[0].axvline(x=seq_len // 2, color='gray', linestyle='--', linewidth=2,
                label='Context Switch')
axes[0].fill_between(range(0, seq_len // 2), 0, max(norms_ungated) * 1.1,
                     alpha=0.05, color='red', label='Context 1')
axes[0].fill_between(range(seq_len // 2, seq_len), 0, max(norms_ungated) * 1.1,
                     alpha=0.05, color='blue', label='Context 2')
axes[0].set_xlabel('Time Step', fontsize=12)
axes[0].set_ylabel('State Frobenius Norm', fontsize=12)
axes[0].set_title('State Magnitude Over Time', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_ylim(bottom=0)

# Right: Gate values
axes[1].plot(range(seq_len), alphas_seq.numpy(), linewidth=2.5, color='#2e7d32',
             label='α (decay gate)', marker='o', markersize=3)
axes[1].plot(range(seq_len), betas_seq.numpy(), linewidth=2.5, color='#f57c00',
             label='β (write strength)', marker='s', markersize=3)
axes[1].axvline(x=seq_len // 2, color='gray', linestyle='--', linewidth=2,
                label='Context Switch')
axes[1].set_xlabel('Time Step', fontsize=12)
axes[1].set_ylabel('Gate Value', fontsize=12)
axes[1].set_title('Gate Values Over Sequence', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_ylim(-0.05, 1.15)

plt.suptitle('Context Switching: Gated vs Ungated', fontsize=15,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left:  The ungated state norm grows monotonically — old context accumulates.")
print("       The gated state drops sharply at the context switch (α ≈ 0),")
print("       then builds back up with the new context.")
print("Right: The gates show when to forget (low α) vs retain (high α).")
print("\n✨ This is exactly the behavior Qwen3.5 learns from data!")

In [ ]:
# Visualize the state matrices at the context switch point
torch.manual_seed(42)
d_viz = 8

# Rebuild states, saving snapshots
S_ungated_snap = torch.zeros(d_viz, d_viz)
S_gated_snap = torch.zeros(d_viz, d_viz)

for t in range(seq_len // 2):  # Process Context 1
    S_ungated_snap = delta_rule_update(S_ungated_snap, keys[t], values[t])
    a, b = alphas_seq[t].item(), betas_seq[t].item()
    decayed = a * S_gated_snap
    pred = decayed.T @ keys[t]
    err = values[t] - pred
    S_gated_snap = decayed + b * torch.outer(keys[t], err)

S_ungated_before = S_ungated_snap.clone()
S_gated_before = S_gated_snap.clone()

# Apply context switch step
t = seq_len // 2
S_ungated_snap = delta_rule_update(S_ungated_snap, keys[t], values[t])
a, b = alphas_seq[t].item(), betas_seq[t].item()
decayed = a * S_gated_snap
pred = decayed.T @ keys[t]
err = values[t] - pred
S_gated_snap = decayed + b * torch.outer(keys[t], err)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
vmax = max(abs(S_ungated_before).max().item(), abs(S_ungated_snap).max().item())

titles = [
    'Ungated: Before Switch', 'Ungated: After Switch',
    'Gated: Before Switch', 'Gated: After Switch (α=0.05)'
]
matrices = [S_ungated_before, S_ungated_snap, S_gated_before, S_gated_snap]
positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

for (r, c), title, mat in zip(positions, titles, matrices):
    im = axes[r, c].imshow(mat.numpy(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[r, c].set_title(f'{title}\n‖S‖ = {torch.norm(mat).item():.2f}',
                          fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=axes[r, c], shrink=0.8)

plt.suptitle('State Matrix at Context Switch', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Top row:    Ungated state barely changes — old context persists.")
print("Bottom row: Gated state is dramatically reduced after the switch (α=0.05).")
print("            The old context has been nearly erased, making room for new information.")

---


In [ ]:
#@title 🎧 Listen: Complementary Roles
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_complementary_roles.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. 🎭 Complementary Roles: Gating vs Delta Rule

It is worth emphasizing that gating and the delta rule do very different things:

| Mechanism | What It Does | Analogy |
|-----------|-------------|---------|
| **Decay gate** ($\alpha_t$) | Scales the *entire* state by a factor | Dimming the lights on the whole whiteboard |
| **Delta rule** + **write gate** ($\beta_t$) | Writes a *targeted* correction at key $k_t$ | Using a marker to fix one specific area |

**Gating = wholesale erasure.** It does not know or care which specific key-value associations are stored. It uniformly decays everything. This is useful for context switches — "forget the cooking discussion, we are doing physics now."

**Delta rule = surgical correction.** It reads what the memory currently predicts for a specific key, computes the error, and writes only the difference. This is useful for incremental updates — "the capital of France used to be stored as Paris, but now I need to update it to Lyon."

**Together, they cover all cases.** The model learns to use low $\alpha$ at topic boundaries and high $\alpha$ within a topic, while the delta rule continuously fine-tunes individual associations.

---

## 4. 🔧 Your Turn: TODO #1 — Implement the Gated Delta Rule Update

Implement `gated_delta_update(S, k, v, alpha, beta)` that computes:

$$S_{\text{new}} = \alpha \cdot S + \beta \cdot k \left(v - (\alpha \cdot S)^T k\right)^T$$

**Important:** The error is computed against the *decayed* state $\alpha \cdot S$, not the original $S$. This is because the decay happens first — we erase, then correct.

Test with:
1. The numerical example from the article ($\alpha = 0.3$, $\beta = 0.9$)
2. Edge case $\alpha = 0$ (total reset)
3. Edge case $\alpha = 1$, $\beta = 1$ (should match the pure delta rule)

In [ ]:
# ============ TODO ============
# Implement the gated delta rule update.
#
# Given:
#   S:     state matrix, shape (d_k, d_v)
#   k:     key vector, shape (d_k,)
#   v:     value vector, shape (d_v,)
#   alpha: decay gate, scalar in (0, 1)
#   beta:  write strength, scalar in (0, 1)
#
# Return:
#   S_new: updated state matrix, shape (d_k, d_v)
#
# Formula: S_new = alpha * S + beta * k @ (v - (alpha * S)^T @ k)^T
#
# Steps:
#   1. Decay the state: S_decayed = alpha * S
#   2. Compute prediction from decayed state: pred = S_decayed^T @ k
#   3. Compute error: error = v - pred
#   4. Write scaled correction: S_new = S_decayed + beta * outer(k, error)
# ============ TODO ============

def gated_delta_update(S: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                       alpha: float, beta: float) -> torch.Tensor:
    """Apply one step of the gated delta rule.

    Args:
        S:     Current state matrix, shape (d_k, d_v)
        k:     Key vector, shape (d_k,)
        v:     Value vector, shape (d_v,)
        alpha: Decay gate ∈ (0, 1) — how much old state to retain
        beta:  Write strength ∈ (0, 1) — how strongly to write the correction

    Returns:
        Updated state matrix S_new, shape (d_k, d_v)
    """
    # Step 1: Decay the state
    S_decayed = ???  # YOUR CODE HERE

    # Step 2: Predict from decayed state
    prediction = ???  # YOUR CODE HERE

    # Step 3: Compute the error
    error = ???  # YOUR CODE HERE

    # Step 4: Write the scaled correction
    S_new = ???  # YOUR CODE HERE

    return S_new

In [ ]:
#@title 🎧 Code Walkthrough: Todo Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_todo_1_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============

S_0 = torch.tensor([[0.8, 0.2],
                     [0.1, 0.9]])
k_1 = torch.tensor([1.0, 0.0])
v_1 = torch.tensor([0.5, 0.3])

try:
    # --- Test 1: Article example (α=0.3, β=0.9) ---
    print("--- Test 1: Article Example (α=0.3, β=0.9) ---")
    S_1 = gated_delta_update(S_0, k_1, v_1, alpha=0.3, beta=0.9)
    expected_S1 = torch.tensor([[0.474, 0.276],
                                [0.030, 0.270]])
    print(f"S_1 (your result):\n{S_1}")
    print(f"S_1 (expected):\n{expected_S1}")
    assert torch.allclose(S_1, expected_S1, atol=1e-3), \
        f"S_1 mismatch! Got\n{S_1}\nExpected\n{expected_S1}"
    print("✅ Article example matches!\n")

    # --- Test 2: Total reset (α=0, β=1) ---
    print("--- Test 2: Total Reset (α=0, β=1) ---")
    S_reset = gated_delta_update(S_0, k_1, v_1, alpha=0.0, beta=1.0)
    # With α=0: S_decayed = 0, pred = 0, error = v, S_new = outer(k, v)
    expected_reset = torch.outer(k_1, v_1)
    print(f"S_reset:\n{S_reset}")
    print(f"Expected (just outer(k, v)):\n{expected_reset}")
    assert torch.allclose(S_reset, expected_reset, atol=1e-6), "Total reset failed!"
    print("✅ Total reset works!\n")

    # --- Test 3: Pure delta rule (α=1, β=1) ---
    print("--- Test 3: Pure Delta Rule (α=1, β=1) ---")
    S_delta = gated_delta_update(S_0, k_1, v_1, alpha=1.0, beta=1.0)
    S_delta_ref = delta_rule_update(S_0, k_1, v_1)
    print(f"Gated (α=1, β=1):\n{S_delta}")
    print(f"Pure delta rule:\n{S_delta_ref}")
    assert torch.allclose(S_delta, S_delta_ref, atol=1e-6), \
        "α=1, β=1 should match pure delta rule!"
    print("✅ Matches pure delta rule!\n")

    # --- Test 4: No update (α=1, β=0) ---
    print("--- Test 4: No Update (α=1, β=0) ---")
    S_noop = gated_delta_update(S_0, k_1, v_1, alpha=1.0, beta=0.0)
    print(f"S_noop:\n{S_noop}")
    assert torch.allclose(S_noop, S_0, atol=1e-6), "α=1, β=0 should pass state through!"
    print("✅ State passes through unchanged!\n")

    print("✅ All 4 tests passed! Your gated delta update is correct.")

except NameError:
    print("❌ Replace the ??? placeholders with your code.")
except Exception as e:
    print(f"❌ Error: {e}")

---
### ✋ Stop and Think
Before looking at the solution:
1. Why do we compute the error against the *decayed* state ($\alpha \cdot S$) rather than the original $S$?
2. When $\alpha = 0$ and $\beta = 1$, what does the state become? Why is this useful?
3. If $\alpha = 0.5$ and $\beta = 0.5$, does the model fully remember or fully forget? What behavior does this produce?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Code Walkthrough: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_todo_1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
def gated_delta_update(S: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                       alpha: float, beta: float) -> torch.Tensor:
    """Apply one step of the gated delta rule.

    S_new = alpha * S + beta * k @ (v - (alpha * S)^T @ k)^T
    """
    # Step 1: Decay the state
    S_decayed = alpha * S

    # Step 2: Predict from decayed state
    prediction = S_decayed.T @ k

    # Step 3: Compute the error
    error = v - prediction

    # Step 4: Write the scaled correction
    S_new = S_decayed + beta * torch.outer(k, error)

    return S_new


# Verify all tests
S_0 = torch.tensor([[0.8, 0.2],
                     [0.1, 0.9]])
k_1 = torch.tensor([1.0, 0.0])
v_1 = torch.tensor([0.5, 0.3])

# Article example
S_1 = gated_delta_update(S_0, k_1, v_1, alpha=0.3, beta=0.9)
print(f"Article example S_1:\n{S_1}")
print(f"Norm reduction: {torch.norm(S_0).item():.4f} → {torch.norm(S_1).item():.4f}")

# Total reset
S_reset = gated_delta_update(S_0, k_1, v_1, alpha=0.0, beta=1.0)
print(f"\nTotal reset: S = outer(k, v) =\n{S_reset}")

# Pure delta rule
S_pure = gated_delta_update(S_0, k_1, v_1, alpha=1.0, beta=1.0)
S_ref = delta_rule_update(S_0, k_1, v_1)
print(f"\nα=1, β=1 matches delta rule: {torch.allclose(S_pure, S_ref)}")

print("\n✅ All good! Gated delta update works correctly.")

---


In [ ]:
#@title 🎧 Listen: Full Gated Deltanet Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_full_gated_deltanet_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. 🏗️ The Full GatedDeltaNet Layer

The gated delta update is the **core recurrence**, but a complete GatedDeltaNet layer — as used in Qwen3.5 — wraps it in several additional components. Let us walk through each one.

### Component 1: Input Projections

The layer takes an input $x_t \in \mathbb{R}^{d_{\text{model}}}$ and projects it into five quantities:

$$Q = W_Q x, \quad K = W_K x, \quad V = W_V x, \quad \alpha = \sigma(W_\alpha x + b_\alpha), \quad \beta = \sigma(W_\beta x + b_\beta)$$

- $Q, K \in \mathbb{R}^{d_k}$ are the query and key (used in the delta rule)
- $V \in \mathbb{R}^{d_v}$ is the value (what gets stored)
- $\alpha, \beta \in (0, 1)$ are the gates (from sigmoid)

### Component 2: Short Causal Convolution (Conv1D)

Before the recurrence, the **query** passes through a short 1D convolution (kernel size 4 in Qwen3.5). This gives the model local context — a 4-token window of neighboring information.

Why only the query? Because the query determines *what to look for* in the state. Convolving the query lets it incorporate local patterns (like n-gram features) before consulting the long-range memory.

```python
# Causal Conv1D: pad on the left so it only sees past tokens
self.conv = nn.Conv1d(d_k, d_k, kernel_size=4, padding=3, groups=d_k)
# After conv, slice [:, :, :seq_len] to remove future leakage
```

### Component 3: SiLU Activation

After the Conv1D, the query passes through the **SiLU** (Sigmoid Linear Unit) activation:

$$\text{SiLU}(x) = x \cdot \sigma(x)$$

SiLU is a smooth, non-monotonic gating function. Compared to ReLU, it allows small negative values through (scaled by the sigmoid), which helps preserve gradient flow.

### Component 4: L2 Normalization

Both queries and keys are **L2-normalized** before the recurrence:

$$\tilde{q}_t = \frac{q_t}{\|q_t\|_2}, \qquad \tilde{k}_t = \frac{k_t}{\|k_t\|_2}$$

This is critical for training stability. Without normalization, the outer products $k_t v_t^T$ can have wildly different magnitudes depending on the norms of $k_t$ and $v_t$, causing the state to grow unboundedly. L2 normalization ensures all keys and queries live on the unit sphere.

### Component 5: Multi-Head Structure

Just like standard multi-head attention, GatedDeltaNet splits the computation across $H$ independent heads. Each head has its own:
- Projections for $Q, K, V, \alpha, \beta$
- State matrix $S_h \in \mathbb{R}^{d_k \times d_v}$
- Independent gated delta updates

The outputs of all heads are concatenated and projected back to $d_{\text{model}}$.

### Complete Data Flow

```
x_t → [W_Q, W_K, W_V, W_α, W_β] → Q, K, V, α_raw, β_raw
                                      │     │
                                      │     ├── sigmoid → α, β
                                      │
                                      Q → Conv1D → SiLU → L2_norm → Q̃
                                      K → L2_norm → K̃
                                      │
                                      └──── Gated Delta Update ────→ output
                                              S_t = α·S_{t-1} + β·K̃(V - S_{t-1}^T K̃)^T
                                              out = S_t^T Q̃
```

In [ ]:
#@title 🎧 What to Look For: Silu Component
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_silu_component.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate each component individually

# --- Component 1: SiLU activation ---
print("=" * 60)
print("  Component: SiLU Activation")
print("=" * 60)

x_range = torch.linspace(-5, 5, 200)
silu_vals = F.silu(x_range)
relu_vals = F.relu(x_range)
sigmoid_vals = torch.sigmoid(x_range)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_range.numpy(), silu_vals.numpy(), linewidth=2.5, color='#1565c0', label='SiLU(x) = x·σ(x)')
ax.plot(x_range.numpy(), relu_vals.numpy(), linewidth=2, color='#e53935', alpha=0.6, label='ReLU(x)')
ax.plot(x_range.numpy(), x_range.numpy() * sigmoid_vals.numpy(), linewidth=2,
        color='#1565c0', alpha=0.0)  # Hidden; same as SiLU
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('Activation(x)', fontsize=12)
ax.set_title('SiLU vs ReLU', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("SiLU allows small negative values through (the dip below 0).")
print("This helps gradient flow compared to ReLU's hard cutoff at 0.")

In [ ]:
#@title 🎧 Code Walkthrough: L2 Norm Component
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_l2_norm_component.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# --- Component 2: L2 Normalization ---
print("=" * 60)
print("  Component: L2 Normalization")
print("=" * 60)

torch.manual_seed(42)
d = 8

# Generate random vectors with different magnitudes
vectors = torch.randn(5, d)
vectors = vectors * torch.tensor([0.1, 1.0, 5.0, 10.0, 50.0]).unsqueeze(1)

print("Before L2 normalization:")
for i, v in enumerate(vectors):
    print(f"  v[{i}]: norm = {v.norm().item():.4f}")

# L2 normalize
vectors_normed = F.normalize(vectors, p=2, dim=1)

print("\nAfter L2 normalization:")
for i, v in enumerate(vectors_normed):
    print(f"  v[{i}]: norm = {v.norm().item():.4f}")

print("\n✅ All vectors now have unit norm.")
print("This is critical: without normalization, the outer products k·v^T")
print("would have wildly different magnitudes, destabilizing the state.")

In [ ]:
#@title 🎧 Code Walkthrough: Conv1d Component
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_conv1d_component.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# --- Component 3: Causal Conv1D ---
print("=" * 60)
print("  Component: Causal Conv1D")
print("=" * 60)

# Demonstrate causal convolution
batch, seq_len, d_model = 1, 8, 4
conv_size = 4

# Create a simple 1D convolution with causal padding
# For causal: pad (kernel_size - 1) on the left, 0 on the right
conv = nn.Conv1d(d_model, d_model, kernel_size=conv_size,
                 padding=conv_size - 1, groups=d_model, bias=False)

# Make weights identifiable: set to ones (averaging filter)
with torch.no_grad():
    conv.weight.fill_(0.25)  # Average of 4 elements

# Input: each position has a distinct pattern
x = torch.arange(1, seq_len + 1, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
x = x.expand(batch, d_model, seq_len)  # (1, 4, 8)

# Apply conv: input shape is (batch, channels, seq_len)
y = conv(x)[:, :, :seq_len]  # Trim to causal (remove future padding)

print(f"Input sequence:  {x[0, 0].tolist()}")
print(f"After Conv1D:    {y[0, 0].tolist()}")
print(f"\nKernel size: {conv_size}, causal padding: {conv_size - 1}")
print(f"Each output position averages the current and {conv_size - 1} previous positions.")
print(f"\nExample: position 4 = avg(1, 2, 3, 4) = {(1+2+3+4)/4}")
print(f"         position 8 = avg(5, 6, 7, 8) = {(5+6+7+8)/4}")
print(f"\n✨ Causal = no future leakage. Each position only sees past + current.")

---

## 6. 🔧 Your Turn: TODO #2 — Build the Complete GatedDeltaNet Layer

Implement a PyTorch `nn.Module` called `GatedDeltaNetLayer` that combines all the components:

1. **Input projections**: Linear layers for Q, K, V, $\alpha$, $\beta$
2. **Causal Conv1D** on Q (kernel size 4)
3. **SiLU activation** on Q (after conv)
4. **L2 normalization** on Q and K
5. **Sigmoid gates** for $\alpha$ and $\beta$
6. **Multi-head gated delta update**: each head maintains independent state
7. **Output projection**: concatenate heads, project back to $d_{\text{model}}$

The forward pass should process a full sequence token-by-token (recurrent mode).

In [ ]:
# ============ TODO ============
# Implement the full GatedDeltaNet layer.
#
# Architecture:
#   Input x: (batch, seq_len, d_model)
#   → Project to Q, K, V, α_raw, β_raw
#   → Reshape for multi-head: (batch, n_heads, seq_len, d_head)
#   → Apply Conv1D on Q (causal, kernel_size=conv_size)
#   → Apply SiLU on Q
#   → L2 normalize Q and K
#   → Apply sigmoid on α_raw and β_raw → α, β per head per timestep
#   → For each timestep, for each head: gated delta update
#   → Concatenate heads, output projection
#
# Shapes:
#   d_model = n_heads * d_k (= n_heads * d_v for simplicity)
#   Q, K: (batch, n_heads, seq_len, d_k)
#   V:    (batch, n_heads, seq_len, d_v)
#   α, β: (batch, n_heads, seq_len, 1)  — per head per timestep
#   S:    (batch, n_heads, d_k, d_v)  — hidden state per head
# ============ TODO ============

class GatedDeltaNetLayer(nn.Module):
    """Complete Gated DeltaNet layer with all components."""

    def __init__(self, d_model: int, n_heads: int, d_k: int, d_v: int,
                 conv_size: int = 4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_k
        self.d_v = d_v
        self.conv_size = conv_size

        # --- Input projections ---
        # Project d_model → n_heads * d_k for Q and K
        # Project d_model → n_heads * d_v for V
        # Project d_model → n_heads for alpha and beta (one gate per head)
        self.W_Q = nn.Linear(d_model, n_heads * d_k, bias=False)
        self.W_K = nn.Linear(d_model, n_heads * d_k, bias=False)
        self.W_V = nn.Linear(d_model, n_heads * d_v, bias=False)
        self.W_alpha = nn.Linear(d_model, n_heads, bias=True)
        self.W_beta = nn.Linear(d_model, n_heads, bias=True)

        # --- Causal Conv1D on Q ---
        # Groups = n_heads * d_k so each channel is convolved independently
        ???  # YOUR CODE HERE: self.conv_q = nn.Conv1d(...)

        # --- Output projection ---
        self.W_out = nn.Linear(n_heads * d_v, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Process a sequence through the GatedDeltaNet layer.

        Args:
            x: Input tensor, shape (batch, seq_len, d_model)

        Returns:
            Output tensor, shape (batch, seq_len, d_model)
        """
        batch, seq_len, _ = x.shape

        # --- Step 1: Input projections ---
        Q = self.W_Q(x)  # (batch, seq_len, n_heads * d_k)
        K = self.W_K(x)  # (batch, seq_len, n_heads * d_k)
        V = self.W_V(x)  # (batch, seq_len, n_heads * d_v)
        alpha_raw = self.W_alpha(x)  # (batch, seq_len, n_heads)
        beta_raw = self.W_beta(x)    # (batch, seq_len, n_heads)

        # --- Step 2: Causal Conv1D on Q ---
        # Conv1d expects (batch, channels, seq_len)
        Q_conv = Q.transpose(1, 2)  # (batch, n_heads * d_k, seq_len)
        Q_conv = self.conv_q(Q_conv)[:, :, :seq_len]  # Causal: trim future
        Q = Q_conv.transpose(1, 2)  # Back to (batch, seq_len, n_heads * d_k)

        # --- Step 3: SiLU activation on Q ---
        ???  # YOUR CODE HERE: apply SiLU to Q

        # --- Step 4: Reshape for multi-head ---
        Q = Q.view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, seq_len, self.n_heads, self.d_v).transpose(1, 2)
        # Q, K: (batch, n_heads, seq_len, d_k)
        # V:    (batch, n_heads, seq_len, d_v)

        # --- Step 5: L2 normalize Q and K ---
        ???  # YOUR CODE HERE: L2 normalize Q and K along the last dimension

        # --- Step 6: Compute gates ---
        alpha = torch.sigmoid(alpha_raw)  # (batch, seq_len, n_heads)
        beta = torch.sigmoid(beta_raw)
        # Reshape to (batch, n_heads, seq_len, 1) for broadcasting
        alpha = alpha.transpose(1, 2).unsqueeze(-1)
        beta = beta.transpose(1, 2).unsqueeze(-1)

        # --- Step 7: Recurrent gated delta update ---
        # Initialize state: one d_k × d_v matrix per batch element per head
        S = torch.zeros(batch, self.n_heads, self.d_k, self.d_v,
                        device=x.device, dtype=x.dtype)
        outputs = []

        for t in range(seq_len):
            k_t = K[:, :, t, :]  # (batch, n_heads, d_k)
            v_t = V[:, :, t, :]  # (batch, n_heads, d_v)
            q_t = Q[:, :, t, :]  # (batch, n_heads, d_k)
            a_t = alpha[:, :, t, :]  # (batch, n_heads, 1)
            b_t = beta[:, :, t, :]   # (batch, n_heads, 1)

            # Gated delta update (batched over batch and heads)
            # S_decayed = alpha * S
            ???  # YOUR CODE HERE: S_decayed = ...

            # prediction = S_decayed^T @ k (batched)
            # S_decayed: (batch, n_heads, d_k, d_v)
            # k_t: (batch, n_heads, d_k)
            # We need: for each (batch, head), compute S_decayed^T @ k
            # = (d_v, d_k) @ (d_k,) = (d_v,)
            # Using einsum: prediction = einsum('bhkv,bhk->bhv', S_decayed, k_t)
            ???  # YOUR CODE HERE: prediction = ...

            # error = v - prediction
            ???  # YOUR CODE HERE: error = ...

            # correction = beta * outer(k, error)
            # outer(k, error): (batch, n_heads, d_k, d_v)
            ???  # YOUR CODE HERE: correction = ...

            # S = S_decayed + correction
            ???  # YOUR CODE HERE: S = ...

            # output = S^T @ q
            # = einsum('bhkv,bhk->bhv', S, q_t)
            out_t = torch.einsum('bhkv,bhk->bhv', S, q_t)  # (batch, n_heads, d_v)
            outputs.append(out_t)

        # --- Step 8: Concatenate and project ---
        # Stack outputs: (batch, n_heads, seq_len, d_v)
        output = torch.stack(outputs, dim=2)
        # Reshape: (batch, seq_len, n_heads * d_v)
        output = output.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        # Project to d_model
        output = self.W_out(output)

        return output

In [ ]:
#@title 🎧 Code Walkthrough: Todo Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_todo_2_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============

try:
    # Create a layer with Qwen3.5-like dimensions (scaled down)
    d_model = 128
    n_heads = 4
    d_k = d_model // n_heads  # = 32
    d_v = d_model // n_heads  # = 32
    conv_size = 4

    layer = GatedDeltaNetLayer(d_model, n_heads, d_k, d_v, conv_size)

    # Count parameters
    total_params = sum(p.numel() for p in layer.parameters())
    print(f"Layer configuration:")
    print(f"  d_model={d_model}, n_heads={n_heads}, d_k={d_k}, d_v={d_v}")
    print(f"  conv_size={conv_size}")
    print(f"  Total parameters: {total_params:,}")

    # Test forward pass
    batch, seq_len = 2, 16
    x = torch.randn(batch, seq_len, d_model)
    output = layer(x)

    print(f"\nInput shape:  {x.shape}")
    print(f"Output shape: {output.shape}")
    assert output.shape == x.shape, f"Expected {x.shape}, got {output.shape}"
    print("✅ Output shape matches input shape!")

    # Check that output contains gradients
    loss = output.sum()
    loss.backward()
    grad_norm = sum(p.grad.norm().item() for p in layer.parameters() if p.grad is not None)
    print(f"\nGradient norm: {grad_norm:.4f}")
    assert grad_norm > 0, "Gradients should be non-zero!"
    print("✅ Gradients flow through the layer!")

    # Check with different sequence lengths
    for sl in [1, 8, 32, 64]:
        layer.zero_grad()
        x_test = torch.randn(1, sl, d_model)
        out_test = layer(x_test)
        assert out_test.shape == (1, sl, d_model), f"Failed for seq_len={sl}"
    print("✅ Works with various sequence lengths (1, 8, 32, 64)!")

    print("\n✅ All checks passed! Your GatedDeltaNet layer is correct.")

except NameError:
    print("❌ Replace the ??? placeholders with your code.")
except Exception as e:
    import traceback
    print(f"❌ Error: {e}")
    traceback.print_exc()

---
### ✋ Stop and Think
Before looking at the solution:
1. Why is the Conv1D applied only to Q and not to K or V?
2. Why do we use `groups=n_heads * d_k` in the Conv1D? What would happen with `groups=1`?
3. The state matrix $S$ has shape $(d_k, d_v)$ per head. How many total floating-point numbers does the state require for a model with 64 heads and $d_k = d_v = 128$?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Code Walkthrough: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_todo_2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
class GatedDeltaNetLayer(nn.Module):
    """Complete Gated DeltaNet layer with all components."""

    def __init__(self, d_model: int, n_heads: int, d_k: int, d_v: int,
                 conv_size: int = 4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_k
        self.d_v = d_v
        self.conv_size = conv_size

        # Input projections
        self.W_Q = nn.Linear(d_model, n_heads * d_k, bias=False)
        self.W_K = nn.Linear(d_model, n_heads * d_k, bias=False)
        self.W_V = nn.Linear(d_model, n_heads * d_v, bias=False)
        self.W_alpha = nn.Linear(d_model, n_heads, bias=True)
        self.W_beta = nn.Linear(d_model, n_heads, bias=True)

        # Causal Conv1D on Q (depthwise: each channel independently)
        self.conv_q = nn.Conv1d(
            n_heads * d_k, n_heads * d_k,
            kernel_size=conv_size,
            padding=conv_size - 1,  # Left-pad for causal
            groups=n_heads * d_k,   # Depthwise
            bias=False
        )

        # Output projection
        self.W_out = nn.Linear(n_heads * d_v, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, _ = x.shape

        # Step 1: Input projections
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        alpha_raw = self.W_alpha(x)
        beta_raw = self.W_beta(x)

        # Step 2: Causal Conv1D on Q
        Q_conv = Q.transpose(1, 2)
        Q_conv = self.conv_q(Q_conv)[:, :, :seq_len]
        Q = Q_conv.transpose(1, 2)

        # Step 3: SiLU activation on Q
        Q = F.silu(Q)

        # Step 4: Reshape for multi-head
        Q = Q.view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, seq_len, self.n_heads, self.d_v).transpose(1, 2)

        # Step 5: L2 normalize Q and K
        Q = F.normalize(Q, p=2, dim=-1)
        K = F.normalize(K, p=2, dim=-1)

        # Step 6: Compute gates
        alpha = torch.sigmoid(alpha_raw).transpose(1, 2).unsqueeze(-1)
        beta = torch.sigmoid(beta_raw).transpose(1, 2).unsqueeze(-1)

        # Step 7: Recurrent gated delta update
        S = torch.zeros(batch, self.n_heads, self.d_k, self.d_v,
                        device=x.device, dtype=x.dtype)
        outputs = []

        for t in range(seq_len):
            k_t = K[:, :, t, :]    # (batch, n_heads, d_k)
            v_t = V[:, :, t, :]    # (batch, n_heads, d_v)
            q_t = Q[:, :, t, :]    # (batch, n_heads, d_k)
            a_t = alpha[:, :, t, :]  # (batch, n_heads, 1)
            b_t = beta[:, :, t, :]   # (batch, n_heads, 1)

            # Decay: S_decayed = alpha * S
            # a_t is (batch, n_heads, 1) — broadcast over (d_k, d_v)
            S_decayed = a_t.unsqueeze(-1) * S  # (batch, n_heads, d_k, d_v)

            # Prediction: S_decayed^T @ k_t
            prediction = torch.einsum('bhkv,bhk->bhv', S_decayed, k_t)

            # Error
            error = v_t - prediction  # (batch, n_heads, d_v)

            # Correction: beta * outer(k_t, error)
            correction = b_t.unsqueeze(-1) * torch.einsum('bhk,bhv->bhkv', k_t, error)

            # Update state
            S = S_decayed + correction

            # Output: S^T @ q_t
            out_t = torch.einsum('bhkv,bhk->bhv', S, q_t)
            outputs.append(out_t)

        # Step 8: Concatenate and project
        output = torch.stack(outputs, dim=2)
        output = output.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        output = self.W_out(output)

        return output


# Test the solution
d_model = 128
n_heads = 4
d_k = d_model // n_heads
d_v = d_model // n_heads
layer = GatedDeltaNetLayer(d_model, n_heads, d_k, d_v, conv_size=4)

# Forward pass
batch, seq_len = 2, 16
x = torch.randn(batch, seq_len, d_model)
output = layer(x)

print(f"GatedDeltaNet Layer:")
print(f"  Input:  {x.shape}")
print(f"  Output: {output.shape}")
print(f"  Params: {sum(p.numel() for p in layer.parameters()):,}")

# Verify gradients
loss = output.sum()
loss.backward()
print(f"  Gradient check: ✅" if all(
    p.grad is not None and p.grad.norm() > 0
    for p in layer.parameters()
) else "  Gradient check: ❌")

print("\n✅ Complete GatedDeltaNet layer works!")

---


In [ ]:
#@title 🎧 Listen: Inference Cost Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_inference_cost_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. ⚡ Inference Cost: O(1) Per Token

One of the most compelling properties of Gated DeltaNet is its **constant-memory, constant-time inference** per token. Let us see exactly why.

### Standard Attention: O(N) Memory and Time

In standard multi-head attention, inference requires storing the full **KV cache** — all past key and value vectors:

$$\text{KV cache size} = 2 \times N \times H \times d_k \times \text{bytes per float}$$

where $N$ is the number of tokens processed so far. For each new token, the model must:
1. Compute attention scores against all $N$ past keys: $O(N \cdot d_k)$ per head
2. Weight and sum all $N$ past values: $O(N \cdot d_v)$ per head
3. Total: $O(N \cdot H \cdot (d_k + d_v))$ per token

As the sequence grows, this becomes a bottleneck — both in memory (the cache grows linearly) and in computation (each new token must attend to all past tokens).

### Gated DeltaNet: O(1) Memory and Time

Gated DeltaNet compresses all past information into a fixed-size state matrix $S \in \mathbb{R}^{d_k \times d_v}$, one per head. For each new token:

1. One matrix-vector product to get prediction: $S^T k$ — $O(d_k \cdot d_v)$
2. One outer product for the update: $k \cdot \text{error}^T$ — $O(d_k \cdot d_v)$
3. One matrix-vector product for output: $S^T q$ — $O(d_k \cdot d_v)$
4. Total: $O(H \cdot d_k \cdot d_v)$ per token — **independent of N!**

### Qwen3.5 Numbers

For Qwen3.5 with $d_k = d_v = 128$ and $H = 64$ heads:

$$\text{State size per layer} = H \times d_k \times d_v \times 4\text{ bytes} = 64 \times 128 \times 128 \times 4 = 4\text{ MB}$$

This is **fixed** regardless of whether the model has processed 100 tokens or 100,000 tokens. Compare to standard attention, which would need:

$$\text{KV cache per layer} = 2 \times N \times H \times d_k \times 4\text{ bytes} = 2 \times N \times 64 \times 128 \times 4$$

At $N = 4096$: $\approx 67$ MB per layer. At $N = 100{,}000$: $\approx 1.6$ GB per layer!

---

## 8. 🔧 Your Turn: TODO #3 — Benchmark GatedDeltaNet vs Standard Attention

Compare inference memory and speed between:

1. **GatedDeltaNet**: process tokens one at a time, maintaining fixed-size state
2. **Standard Attention**: use full KV cache, recomputing attention at each step

For sequence lengths $[128, 512, 1024, 2048, 4096]$, measure:
- Peak memory usage
- Time per token

Then plot both comparisons.

In [ ]:
# ============ TODO ============
# Benchmark GatedDeltaNet vs Standard Attention.
#
# Part A: Implement a simple standard attention forward pass for comparison
#   - Store K, V in a growing cache
#   - For each new token: compute attention scores against all cached K,
#     weight cached V, produce output
#
# Part B: Benchmark both methods
#   - For each seq_len in [128, 512, 1024, 2048, 4096]:
#     1. GatedDeltaNet: process tokens one at a time with the layer above
#     2. Standard attention: process tokens one at a time with KV cache
#     3. Record time per token and peak memory
#
# Part C: Plot the results
#   - Left plot: Memory usage vs sequence length
#   - Right plot: Time per token vs sequence length
# ============ TODO ============

class StandardAttentionInference:
    """Standard multi-head attention with KV cache for inference."""

    def __init__(self, d_model: int, n_heads: int):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = torch.randn(d_model, d_model) * 0.02
        self.W_K = torch.randn(d_model, d_model) * 0.02
        self.W_V = torch.randn(d_model, d_model) * 0.02
        self.W_O = torch.randn(d_model, d_model) * 0.02
        self.k_cache = []  # List of (n_heads, d_k) tensors
        self.v_cache = []

    def reset(self):
        self.k_cache = []
        self.v_cache = []

    @torch.no_grad()
    def forward_one_token(self, x: torch.Tensor) -> torch.Tensor:
        """Process one token with KV cache.

        Args:
            x: (d_model,) — single token embedding

        Returns:
            output: (d_model,) — attention output
        """
        # Project
        q = (x @ self.W_Q).view(self.n_heads, self.d_k)  # (n_heads, d_k)
        k = (x @ self.W_K).view(self.n_heads, self.d_k)
        v = (x @ self.W_V).view(self.n_heads, self.d_k)

        # Append to cache
        self.k_cache.append(k)
        self.v_cache.append(v)

        # Stack cache: (seq_so_far, n_heads, d_k)
        K_cache = torch.stack(self.k_cache, dim=0)  # (t, n_heads, d_k)
        V_cache = torch.stack(self.v_cache, dim=0)

        # Attention scores: q @ K^T / sqrt(d_k)
        # q: (n_heads, d_k), K_cache: (t, n_heads, d_k)
        scores = torch.einsum('hd,thd->ht', q, K_cache) / (self.d_k ** 0.5)
        weights = torch.softmax(scores, dim=-1)  # (n_heads, t)

        # Weighted sum of values
        out = torch.einsum('ht,thd->hd', weights, V_cache)  # (n_heads, d_k)
        out = out.reshape(-1) @ self.W_O  # (d_model,)
        return out


class GatedDeltaNetInference:
    """GatedDeltaNet inference with fixed-size state."""

    def __init__(self, d_model: int, n_heads: int):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_v = d_model // n_heads
        # Random projections (just for benchmarking shape/speed)
        self.W_Q = torch.randn(d_model, n_heads * self.d_k) * 0.02
        self.W_K = torch.randn(d_model, n_heads * self.d_k) * 0.02
        self.W_V = torch.randn(d_model, n_heads * self.d_v) * 0.02
        self.W_alpha = torch.randn(d_model, n_heads) * 0.02
        self.W_beta = torch.randn(d_model, n_heads) * 0.02
        self.W_O = torch.randn(n_heads * self.d_v, d_model) * 0.02
        # State: (n_heads, d_k, d_v) — FIXED SIZE regardless of seq_len
        self.S = torch.zeros(n_heads, self.d_k, self.d_v)

    def reset(self):
        self.S = torch.zeros(self.n_heads, self.d_k, self.d_v)

    @torch.no_grad()
    def forward_one_token(self, x: torch.Tensor) -> torch.Tensor:
        """Process one token with fixed-size state.

        Args:
            x: (d_model,) — single token embedding

        Returns:
            output: (d_model,)
        """
        # Project
        q = (x @ self.W_Q).view(self.n_heads, self.d_k)
        k = (x @ self.W_K).view(self.n_heads, self.d_k)
        v = (x @ self.W_V).view(self.n_heads, self.d_v)
        alpha = torch.sigmoid(x @ self.W_alpha)  # (n_heads,)
        beta = torch.sigmoid(x @ self.W_beta)

        # L2 normalize q, k
        q = F.normalize(q, p=2, dim=-1)
        k = F.normalize(k, p=2, dim=-1)

        # Gated delta update (vectorized over heads)
        # S: (n_heads, d_k, d_v)
        S_decayed = alpha.unsqueeze(-1).unsqueeze(-1) * self.S
        prediction = torch.einsum('hkv,hk->hv', S_decayed, k)
        error = v - prediction
        correction = beta.unsqueeze(-1).unsqueeze(-1) * torch.einsum('hk,hv->hkv', k, error)
        self.S = S_decayed + correction

        # Output
        out = torch.einsum('hkv,hk->hv', self.S, q)  # (n_heads, d_v)
        out = out.reshape(-1) @ self.W_O  # (d_model,)
        return out


# --- Benchmark ---
d_model = 128
n_heads = 4
seq_lengths = [128, 512, 1024, 2048, 4096]

gdn_times = []
std_times = []
gdn_memory = []  # Estimated memory in bytes
std_memory = []

print(f"{'Seq Len':<10} {'StdAttn (ms/tok)':<18} {'GatedDN (ms/tok)':<18} "
      f"{'StdAttn Mem (KB)':<18} {'GatedDN Mem (KB)':<18}")
print("-" * 85)

for seq_len in seq_lengths:
    # Standard Attention
    std_attn = StandardAttentionInference(d_model, n_heads)
    std_attn.reset()
    x_seq = torch.randn(seq_len, d_model)

    start = time.time()
    for t in range(seq_len):
        _ = std_attn.forward_one_token(x_seq[t])
    std_time = (time.time() - start) / seq_len * 1000  # ms per token
    std_times.append(std_time)

    # Memory: KV cache = 2 * seq_len * n_heads * d_k * 4 bytes
    std_mem = ???  # YOUR CODE HERE: compute KV cache memory in KB
    std_memory.append(std_mem)

    # Gated DeltaNet
    gdn = GatedDeltaNetInference(d_model, n_heads)
    gdn.reset()

    start = time.time()
    for t in range(seq_len):
        _ = gdn.forward_one_token(x_seq[t])
    gdn_time = (time.time() - start) / seq_len * 1000  # ms per token
    gdn_times.append(gdn_time)

    # Memory: state = n_heads * d_k * d_v * 4 bytes (CONSTANT)
    gdn_mem = ???  # YOUR CODE HERE: compute state memory in KB
    gdn_memory.append(gdn_mem)

    print(f"{seq_len:<10} {std_time:<18.4f} {gdn_time:<18.4f} "
          f"{std_mem:<18.1f} {gdn_mem:<18.1f}")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Memory
???  # YOUR CODE HERE
# axes[0].plot(seq_lengths, std_memory, ...) for standard attention
# axes[0].plot(seq_lengths, gdn_memory, ...) for GatedDeltaNet
# Add labels, title, legend

# Right: Time per token
???  # YOUR CODE HERE
# axes[1].plot(seq_lengths, std_times, ...) for standard attention
# axes[1].plot(seq_lengths, gdn_times, ...) for GatedDeltaNet
# Add labels, title, legend

plt.tight_layout()
plt.show()

---
### ✋ Stop and Think
Before looking at the solution:
1. Why is GatedDeltaNet memory constant while standard attention memory grows linearly?
2. At what sequence length does the standard attention KV cache exceed the GatedDeltaNet state size?
3. Why might time per token also increase for standard attention? (Hint: what operation scales with N?)

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 What to Look For: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_todo_3_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
# --- Full benchmark solution ---
d_model = 128
n_heads = 4
d_k = d_model // n_heads  # 32
d_v = d_model // n_heads  # 32
seq_lengths = [128, 512, 1024, 2048, 4096]

gdn_times = []
std_times = []
gdn_memory = []
std_memory = []

print(f"{'Seq Len':<10} {'StdAttn (ms/tok)':<18} {'GatedDN (ms/tok)':<18} "
      f"{'StdAttn Mem (KB)':<18} {'GatedDN Mem (KB)':<18}")
print("-" * 85)

for seq_len in seq_lengths:
    # Standard Attention
    std_attn = StandardAttentionInference(d_model, n_heads)
    std_attn.reset()
    x_seq = torch.randn(seq_len, d_model)

    start = time.time()
    for t in range(seq_len):
        _ = std_attn.forward_one_token(x_seq[t])
    std_time = (time.time() - start) / seq_len * 1000
    std_times.append(std_time)

    # KV cache memory: 2 (K + V) * seq_len * n_heads * d_k * 4 bytes / 1024
    std_mem = 2 * seq_len * n_heads * d_k * 4 / 1024  # KB
    std_memory.append(std_mem)

    # Gated DeltaNet
    gdn = GatedDeltaNetInference(d_model, n_heads)
    gdn.reset()

    start = time.time()
    for t in range(seq_len):
        _ = gdn.forward_one_token(x_seq[t])
    gdn_time = (time.time() - start) / seq_len * 1000
    gdn_times.append(gdn_time)

    # State memory: n_heads * d_k * d_v * 4 bytes / 1024 (CONSTANT)
    gdn_mem = n_heads * d_k * d_v * 4 / 1024  # KB
    gdn_memory.append(gdn_mem)

    print(f"{seq_len:<10} {std_time:<18.4f} {gdn_time:<18.4f} "
          f"{std_mem:<18.1f} {gdn_mem:<18.1f}")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Memory
axes[0].plot(seq_lengths, std_memory, 'o-', color='#e53935', linewidth=2.5,
             markersize=8, label='Standard Attention (KV Cache)')
axes[0].plot(seq_lengths, gdn_memory, 's-', color='#1565c0', linewidth=2.5,
             markersize=8, label='GatedDeltaNet (Fixed State)')
axes[0].set_xlabel('Sequence Length (N)', fontsize=12)
axes[0].set_ylabel('Memory (KB)', fontsize=12)
axes[0].set_title('Inference Memory Usage', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_yscale('log')

# Annotate the constant line
axes[0].annotate(f'{gdn_memory[0]:.0f} KB (constant!)',
                 xy=(seq_lengths[-1], gdn_memory[-1]),
                 xytext=(seq_lengths[-1] * 0.5, gdn_memory[-1] * 3),
                 fontsize=10, color='#1565c0', fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='#1565c0'))

# Right: Time per token
axes[1].plot(seq_lengths, std_times, 'o-', color='#e53935', linewidth=2.5,
             markersize=8, label='Standard Attention')
axes[1].plot(seq_lengths, gdn_times, 's-', color='#1565c0', linewidth=2.5,
             markersize=8, label='GatedDeltaNet')
axes[1].set_xlabel('Sequence Length (N)', fontsize=12)
axes[1].set_ylabel('Time per Token (ms)', fontsize=12)
axes[1].set_title('Inference Speed', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('GatedDeltaNet vs Standard Attention: Inference Cost',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 Key observations:")
print(f"  Memory: StdAttn grows linearly (O(N)). GatedDN is constant ({gdn_memory[0]:.0f} KB).")
print(f"  Speed:  StdAttn time per token grows with N (must attend to all past tokens).")
print(f"          GatedDN time per token is roughly constant (fixed-size state update).")
print(f"\n  At N=4096: StdAttn uses {std_memory[-1]:.0f} KB vs GatedDN's {gdn_memory[-1]:.0f} KB")
print(f"  That is a {std_memory[-1]/gdn_memory[-1]:.0f}x difference!")

---


In [ ]:
#@title 🎧 What to Look For: Qwen3 At Scale
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_qwen3_5_at_scale.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. 🔍 Qwen3.5 at Scale

Let us compute the actual numbers for Qwen3.5 to appreciate the practical impact.

In [ ]:
# Qwen3.5 scale analysis
print("=" * 60)
print("  QWEN3.5: Memory Analysis")
print("=" * 60)

# Qwen3.5 parameters (from the article)
d_k_qwen = 128
d_v_qwen = 128
n_heads_qwen = 64
n_layers_gdn = 36  # 75% of 48 layers use GatedDeltaNet
n_layers_attn = 12  # 25% use standard attention
bytes_per_param = 2  # FP16

# GatedDeltaNet state per layer
state_per_head = d_k_qwen * d_v_qwen * bytes_per_param  # bytes
state_per_layer = n_heads_qwen * state_per_head
state_total_gdn = n_layers_gdn * state_per_layer

print(f"\nGatedDeltaNet layers ({n_layers_gdn} layers):")
print(f"  State per head:  {d_k_qwen} × {d_v_qwen} × {bytes_per_param} = {state_per_head / 1024:.1f} KB")
print(f"  State per layer: {n_heads_qwen} heads × {state_per_head / 1024:.1f} KB = {state_per_layer / (1024**2):.1f} MB")
print(f"  Total GDN state: {n_layers_gdn} layers × {state_per_layer / (1024**2):.1f} MB = {state_total_gdn / (1024**2):.0f} MB")
print(f"  → Fixed regardless of sequence length!")

# Standard attention KV cache for comparison
print(f"\nStandard Attention layers ({n_layers_attn} layers):")
for N in [1024, 4096, 16384, 65536, 131072]:
    kv_per_layer = 2 * N * n_heads_qwen * d_k_qwen * bytes_per_param  # K + V
    kv_total = n_layers_attn * kv_per_layer
    total = state_total_gdn + kv_total
    print(f"  N={N:>7}: KV cache = {kv_total / (1024**2):>8.1f} MB | "
          f"Total (GDN + Attn) = {total / (1024**2):>8.1f} MB")

# What if ALL layers were standard attention?
print(f"\nIf ALL {n_layers_gdn + n_layers_attn} layers used standard attention:")
total_layers = n_layers_gdn + n_layers_attn
for N in [1024, 4096, 16384, 65536, 131072]:
    kv_total_all = total_layers * 2 * N * n_heads_qwen * d_k_qwen * bytes_per_param
    hybrid_total = state_total_gdn + n_layers_attn * 2 * N * n_heads_qwen * d_k_qwen * bytes_per_param
    savings = (kv_total_all - hybrid_total) / kv_total_all * 100
    print(f"  N={N:>7}: Full attention = {kv_total_all / (1024**3):>6.1f} GB | "
          f"Hybrid = {hybrid_total / (1024**3):>6.1f} GB | "
          f"Savings = {savings:.0f}%")

# Visualize
seq_lens = np.array([256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])

full_attn_mem = []
hybrid_mem = []
gdn_only = state_total_gdn / (1024**3)  # GB

for N in seq_lens:
    kv_all = total_layers * 2 * N * n_heads_qwen * d_k_qwen * bytes_per_param
    kv_partial = n_layers_attn * 2 * N * n_heads_qwen * d_k_qwen * bytes_per_param
    full_attn_mem.append(kv_all / (1024**3))
    hybrid_mem.append((state_total_gdn + kv_partial) / (1024**3))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(seq_lens, full_attn_mem, 'o-', color='#e53935', linewidth=2.5,
        markersize=6, label=f'All Attention ({total_layers} layers)')
ax.plot(seq_lens, hybrid_mem, 's-', color='#1565c0', linewidth=2.5,
        markersize=6, label=f'Hybrid ({n_layers_gdn} GDN + {n_layers_attn} Attn)')
ax.axhline(y=gdn_only, color='#2e7d32', linestyle='--', linewidth=2,
           label=f'GDN state only ({gdn_only*1024:.0f} MB, constant)')

ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_title('Qwen3.5: KV Cache Memory vs Sequence Length', fontsize=14,
             fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\n✨ The hybrid architecture (75% GDN + 25% Attention) dramatically reduces")
print("   memory requirements at long sequence lengths while retaining the")
print("   precise recall ability of standard attention for the most critical layers.")

---


In [ ]:
#@title 🎧 What to Look For: Multi Head State Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_multi_head_state_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 10. 🎨 Multi-Head State Visualization

Let us look at what the state matrices look like across different heads after processing a sequence. Each head maintains an independent memory, potentially specializing in different types of information.

In [ ]:
# Visualize multi-head states after processing a sequence
torch.manual_seed(42)

d_model = 64
n_heads = 4
d_k = d_model // n_heads  # 16
d_v = d_model // n_heads
layer = GatedDeltaNetLayer(d_model, n_heads, d_k, d_v, conv_size=4)

# Process a random sequence
seq_len = 32
x = torch.randn(1, seq_len, d_model)

# We need to extract the internal states — let's manually run the forward
with torch.no_grad():
    Q = layer.W_Q(x)
    K = layer.W_K(x)
    V = layer.W_V(x)
    alpha_raw = layer.W_alpha(x)
    beta_raw = layer.W_beta(x)

    Q_conv = Q.transpose(1, 2)
    Q_conv = layer.conv_q(Q_conv)[:, :, :seq_len]
    Q = Q_conv.transpose(1, 2)
    Q = F.silu(Q)

    Q = Q.view(1, seq_len, n_heads, d_k).transpose(1, 2)
    K = K.view(1, seq_len, n_heads, d_k).transpose(1, 2)
    V = V.view(1, seq_len, n_heads, d_v).transpose(1, 2)

    Q = F.normalize(Q, p=2, dim=-1)
    K = F.normalize(K, p=2, dim=-1)

    alpha = torch.sigmoid(alpha_raw).transpose(1, 2).unsqueeze(-1)
    beta = torch.sigmoid(beta_raw).transpose(1, 2).unsqueeze(-1)

    S = torch.zeros(1, n_heads, d_k, d_v)
    state_snapshots = []

    for t in range(seq_len):
        k_t = K[:, :, t, :]
        v_t = V[:, :, t, :]
        a_t = alpha[:, :, t, :]
        b_t = beta[:, :, t, :]

        S_decayed = a_t.unsqueeze(-1) * S
        prediction = torch.einsum('bhkv,bhk->bhv', S_decayed, k_t)
        error = v_t - prediction
        correction = b_t.unsqueeze(-1) * torch.einsum('bhk,bhv->bhkv', k_t, error)
        S = S_decayed + correction

        if (t + 1) % 8 == 0:  # Snapshot every 8 steps
            state_snapshots.append(S[0].clone())  # (n_heads, d_k, d_v)

# Plot: 4 heads × 4 time snapshots
fig, axes = plt.subplots(n_heads, len(state_snapshots), figsize=(16, 12))
vmax = max(s.abs().max().item() for s in state_snapshots)

for h in range(n_heads):
    for s_idx, (t_val, snapshot) in enumerate(zip(range(8, seq_len + 1, 8), state_snapshots)):
        mat = snapshot[h].numpy()
        im = axes[h, s_idx].imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[h, s_idx].set_title(f'Head {h}, t={t_val}\n‖S‖={torch.norm(snapshot[h]).item():.2f}',
                                  fontsize=9)
        if h == n_heads - 1:
            axes[h, s_idx].set_xlabel('d_v', fontsize=9)
        if s_idx == 0:
            axes[h, s_idx].set_ylabel(f'Head {h}\nd_k', fontsize=9)

plt.suptitle('Multi-Head State Matrices Over Time', fontsize=15, fontweight='bold', y=1.02)
fig.colorbar(im, ax=axes, shrink=0.6, label='Value')
plt.tight_layout()
plt.show()

print("Each head develops a different state pattern — the heads specialize.")
print("Some heads may focus on recent tokens (low α → frequent resets),")
print("while others maintain longer-term memory (high α → persistent state).")

In [ ]:
# Analyze gate distributions across heads
with torch.no_grad():
    alpha_vals = torch.sigmoid(layer.W_alpha(x)).squeeze(0)  # (seq_len, n_heads)
    beta_vals = torch.sigmoid(layer.W_beta(x)).squeeze(0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Alpha distribution per head (box plot)
alpha_data = [alpha_vals[:, h].numpy() for h in range(n_heads)]
bp1 = axes[0].boxplot(alpha_data, labels=[f'Head {h}' for h in range(n_heads)],
                       patch_artist=True)
colors = ['#e53935', '#1565c0', '#2e7d32', '#f57c00']
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0].set_ylabel('α (decay gate)', fontsize=12)
axes[0].set_title('Decay Gate Distribution Per Head', fontsize=14, fontweight='bold')
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

# Right: Beta distribution per head
beta_data = [beta_vals[:, h].numpy() for h in range(n_heads)]
bp2 = axes[1].boxplot(beta_data, labels=[f'Head {h}' for h in range(n_heads)],
                       patch_artist=True)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_ylabel('β (write strength)', fontsize=12)
axes[1].set_title('Write Strength Distribution Per Head', fontsize=14, fontweight='bold')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Gate Value Distributions (Random Initialization)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Also show gate values over time for each head
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for h in range(n_heads):
    axes[0].plot(range(seq_len), alpha_vals[:, h].numpy(), linewidth=1.5,
                 color=colors[h], label=f'Head {h}', alpha=0.8)
    axes[1].plot(range(seq_len), beta_vals[:, h].numpy(), linewidth=1.5,
                 color=colors[h], label=f'Head {h}', alpha=0.8)

axes[0].set_ylabel('α (decay gate)', fontsize=12)
axes[0].set_title('Gate Values Over Sequence', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9, ncol=4)
axes[0].set_ylim(-0.05, 1.05)

axes[1].set_xlabel('Time Step', fontsize=12)
axes[1].set_ylabel('β (write strength)', fontsize=12)
axes[1].legend(fontsize=9, ncol=4)
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

print("At random initialization, the gates hover near 0.5 (sigmoid of ~0).")
print("After training, different heads would learn different gating patterns:")
print("  - Some heads: high α (long-term memory)")
print("  - Some heads: variable α (adaptive forgetting)")
print("  - Some heads: low α (short-term, frequently reset)")

---

## 11. 🤔 Reflection and Key Takeaways

### What We Built

In this notebook, we went from the delta rule (Notebook 2) to the **complete Gated DeltaNet layer** as used in Qwen3.5. Here is what each component contributes:

| Component | Purpose | Why It Matters |
|-----------|---------|---------------|
| **Decay gate** ($\alpha_t$) | Wholesale memory erasure | Enables context switching — "forget everything" |
| **Write gate** ($\beta_t$) | Controls correction strength | Prevents overwriting when the current prediction is already good |
| **Delta rule** | Error-correcting writes | Prevents superposition, surgically updates individual associations |
| **Conv1D** | Local context for queries | Captures n-gram patterns before consulting long-range memory |
| **SiLU** | Smooth activation | Better gradient flow than ReLU |
| **L2 norm** | Stabilize Q/K magnitudes | Prevents state from growing unboundedly |
| **Multi-head** | Parallel independent memories | Different heads can specialize in different types of information |

### The Two Key Innovations

1. **Gating + delta rule** give the model two complementary tools:
   - $\alpha_t$: "How much of the past should I keep?" (wholesale)
   - Delta rule + $\beta_t$: "What specific correction should I make?" (surgical)

2. **O(1) inference cost** per token:
   - Fixed state matrix instead of growing KV cache
   - For Qwen3.5: **4 MB per layer** vs potentially **GBs** for standard attention at long sequences

### Connection to Qwen3.5

The Gated DeltaNet layer we built is the actual layer used in 75% of Qwen3.5's layers (36 out of 48). The remaining 25% (12 layers) use standard sliding-window attention for precise token-level recall when it is needed.

This **hybrid architecture** gets the best of both worlds:
- **GatedDeltaNet layers**: Efficient long-range context compression ($O(1)$ per token)
- **Standard attention layers**: Precise recall for recent context (within a sliding window)

### Reflection Questions

1. **Why does Qwen3.5 still keep 25% standard attention layers?** What can standard attention do that GatedDeltaNet cannot?

2. **The gates $\alpha_t$ and $\beta_t$ are computed per-head.** Why is this better than a single global gate for the whole layer?

3. **The Conv1D has kernel size 4.** Why not larger? What is the tradeoff between local context window size and computational cost?

4. **Could you train the gates to be binary (0 or 1) instead of continuous?** What would the training difficulties be?

---


In [ ]:
#@title 🎧 Transition: Next Steps
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_next_steps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 12. 🚀 What Comes Next

In **Notebook 4: Hybrid Attention + Mixture of Experts**, we will:

1. **Build the hybrid architecture** — interleaving GatedDeltaNet and standard sliding-window attention layers
2. **Implement Mixture of Experts (MoE)** — the other major efficiency innovation in Qwen3.5, where each token is routed to only 8 of 128 experts
3. **Put it all together** — assemble a mini Qwen3.5 block with hybrid attention + MoE and trace a token through the complete forward pass
4. **Analyze the compute savings** — how the combination of GDN + MoE achieves massive efficiency gains

We now have the two building blocks: efficient memory (GatedDeltaNet) and precise recall (standard attention). Next, we learn how to combine them and add conditional computation.

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_28_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Final summary
print("""
╔════════════════════════════════════════════════════════════╗
║           GATED DELTANET CHEAT SHEET                      ║
╠════════════════════════════════════════════════════════════╣
║                                                            ║
║  Update Rule:                                              ║
║    S_t = α_t · S_{t-1}                                     ║
║        + β_t · k_t (v_t - (α_t · S_{t-1})^T k_t)^T        ║
║                                                            ║
║  Components:                                               ║
║    α_t = σ(W_α x_t + b_α)    — decay gate (forget)        ║
║    β_t = σ(W_β x_t + b_β)    — write gate (update)        ║
║    Q  → Conv1D → SiLU → L2 norm                           ║
║    K  → L2 norm                                            ║
║    V  → direct to recurrence                               ║
║                                                            ║
║  Inference Cost:                                           ║
║    Memory: O(H · d_k · d_v) per layer — CONSTANT           ║
║    Time:   O(H · d_k · d_v) per token — CONSTANT           ║
║    Qwen3.5: 4 MB/layer (vs GBs for standard attention)     ║
║                                                            ║
║  Key Insight:                                              ║
║    Gating = wholesale erasure (sledgehammer)                ║
║    Delta  = surgical correction (scalpel)                   ║
║    Together = full memory management                        ║
║                                                            ║
║  In Qwen3.5:                                               ║
║    75% GatedDeltaNet layers (efficient long-range)          ║
║    25% Standard Attention layers (precise recall)           ║
║                                                            ║
╚════════════════════════════════════════════════════════════╝
""")
print("✅ Notebook complete! You now understand the full Gated DeltaNet layer.")
print("🚀 Next up: Notebook 4 — Hybrid Attention + Mixture of Experts")